# Customer Churn Prediction

**Goal:** Predict which telecom customers are likely to cancel their subscription (churn) so the business can proactively retain them.

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 features)

**Pipeline:**
1. Data Cleaning & EDA
2. Feature Engineering (encoding, scaling, train/test split)
3. Model Training (Logistic Regression, Random Forest, XGBoost)
4. Evaluation (Confusion Matrix, ROC, Precision-Recall, Feature Importance)
5. Hyperparameter Tuning (GridSearchCV, F1-optimized)
6. Interpretation (SHAP) & Business Recommendations

In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, auc, precision_recall_curve,
                             average_precision_score, roc_auc_score)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Paths
DATA_DIR = 'data/'
MODELS_DIR = 'models/'
ASSETS_DIR = 'assets/'
RAW_DATA = DATA_DIR + 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

---
## Step 1: Data Loading & Cleaning

In [ ]:
# Load raw data
df = pd.read_csv(RAW_DATA)
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

# Drop customerID (unique identifier, no predictive value)
df.drop(columns=['customerID'], inplace=True)
print('Dropped customerID')

# TotalCharges: blanks -> NaN -> fill with median
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
null_count = df['TotalCharges'].isna().sum()
if null_count > 0:
    median_val = df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_val)
    print(f'TotalCharges: filled {null_count} blanks with median ({median_val})')

# SeniorCitizen: cast to categorical (binary 0/1 but is a category)
df['SeniorCitizen'] = df['SeniorCitizen'].astype(object)
print('Casted SeniorCitizen to categorical')

## Step 2: Target Encoding

In [ ]:
# Map Churn: Yes -> 1, No -> 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f'Churn distribution:\n{df["Churn"].value_counts()}')
print(f'Baseline churn rate: {df["Churn"].mean():.3f} ({df["Churn"].mean()*100:.1f}%)')

## Step 3: One-Hot Encoding

In [ ]:
# Identify categorical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {cat_cols}')

# One-hot encode with drop_first to avoid multicollinearity
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert bool columns to int
for col in df_encoded.select_dtypes('bool').columns:
    df_encoded[col] = df_encoded[col].astype(int)

print(f'Encoded shape: {df_encoded.shape}')
print(f'All numeric: {all(df_encoded.dtypes.apply(pd.api.types.is_numeric_dtype))}')

## Step 4: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 4a. Target distribution
sns.countplot(x='Churn', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of Churn (Target)')
axes[0].set_xticklabels(['No (0)', 'Yes (1)'])
for p in axes[0].containers:
    axes[0].bar_label(p)

# 4b. Churn rate by Contract
churn_by_contract = df.groupby('Contract')['Churn'].mean().reset_index()
sns.barplot(x='Contract', y='Churn', data=churn_by_contract, ax=axes[1],
            palette='rocket', order=['Month-to-month', 'One year', 'Two year'])
axes[1].set_title('Churn Rate by Contract Type')
axes[1].set_ylabel('Mean Churn Rate')
axes[1].set_ylim(0, 1)

# 4c. Tenure vs Churn
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[2], palette='Set2')
axes[2].set_title('Tenure (Months) vs. Churn')
axes[2].set_xticklabels(['No (0)', 'Yes (1)'])

plt.tight_layout()
plt.savefig(ASSETS_DIR + 'eda_churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/eda_churn_analysis.png')

## Step 5: Export Cleaned Data

In [ ]:
df_encoded.to_csv(DATA_DIR + 'cleaned_churn_data.csv', index=False)
print(f'Exported: {DATA_DIR}cleaned_churn_data.csv ({df_encoded.shape})')

---
## Step 6: Feature Engineering (Split & Scale)

In [ ]:
# Separate features and target
y = df_encoded['Churn']
X = df_encoded.drop(columns=['Churn'])
feature_names = X.columns.tolist()

# Stratified 80/20 split (preserves churn ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')
print(f'Train churn rate: {y_train.mean():.3f}  |  Test churn rate: {y_test.mean():.3f}')

# StandardScaler (fit on train only to prevent data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names, index=X_test.index)

print(f'Post-scale: mean ~ {X_train_scaled.mean().mean():.4f}, std ~ {X_train_scaled.std().mean():.4f}')

# Save splits
X_train_scaled.to_csv(DATA_DIR + 'X_train.csv', index=False)
X_test_scaled.to_csv(DATA_DIR + 'X_test.csv', index=False)
y_train.to_csv(DATA_DIR + 'y_train.csv', index=False)
y_test.to_csv(DATA_DIR + 'y_test.csv', index=False)
print('Saved train/test splits to data/')

---
## Step 7: Model Training (Default Hyperparameters)

Training 3 models to establish baselines:

In [ ]:
models = {}

# 7a. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr
y_pred_lr = lr.predict(X_test_scaled)
print('Logistic Regression:')
print(f'  Acc={accuracy_score(y_test, y_pred_lr):.3f}, Prec={precision_score(y_test, y_pred_lr):.3f}, '
      f'Recall={recall_score(y_test, y_pred_lr):.3f}, F1={f1_score(y_test, y_pred_lr):.3f}')

# 7b. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
models['Random Forest'] = rf
y_pred_rf = rf.predict(X_test_scaled)
print('Random Forest:')
print(f'  Acc={accuracy_score(y_test, y_pred_rf):.3f}, Prec={precision_score(y_test, y_pred_rf):.3f}, '
      f'Recall={recall_score(y_test, y_pred_rf):.3f}, F1={f1_score(y_test, y_pred_rf):.3f}')

# 7c. XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                               random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)
models['XGBoost'] = xgb_model
y_pred_xgb = xgb_model.predict(X_test_scaled)
print('XGBoost:')
print(f'  Acc={accuracy_score(y_test, y_pred_xgb):.3f}, Prec={precision_score(y_test, y_pred_xgb):.3f}, '
      f'Recall={recall_score(y_test, y_pred_xgb):.3f}, F1={f1_score(y_test, y_pred_xgb):.3f}')

# Save default models
joblib.dump(lr, MODELS_DIR + 'model_logistic_regression.pkl')
joblib.dump(rf, MODELS_DIR + 'model_random_forest.pkl')
joblib.dump(xgb_model, MODELS_DIR + 'model_xgboost.pkl')
print('\nDefault models saved to models/')

---
## Step 8: Model Evaluation

We evaluate models on 4 plots: Confusion Matrix, ROC Curve, Precision-Recall Curve, and Feature Importance.

In [ ]:
# 8a. Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticklabels(['No Churn', 'Churn'])
    ax.set_yticklabels(['No Churn', 'Churn'])
plt.tight_layout()
plt.savefig(ASSETS_DIR + 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/confusion_matrices.png')

In [ ]:
# 8b. ROC Curves
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR + 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/roc_curves.png')

In [ ]:
# 8c. Precision-Recall Curves
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    plt.plot(recall, precision, label=f'{name} (AP = {ap:.3f})')
plt.axhline(y=y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR + 'precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/precision_recall_curves.png')

In [ ]:
# 8d. Feature Importance
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

# LogReg coefficients
coefs = pd.Series(lr.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
top_coef = coefs.head(15)
colors = ['red' if v < 0 else 'green' for v in top_coef.values]
top_coef.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Logistic Regression (Top 15)')
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_xlabel('Coefficient')

# RF importance
rf_imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
rf_imp.tail(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Random Forest (Top 15)')

# XGBoost importance
xgb_imp = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=True)
xgb_imp.tail(15).plot(kind='barh', ax=axes[2], color='darkorange')
axes[2].set_title('XGBoost (Top 15)')

plt.tight_layout()
plt.savefig(ASSETS_DIR + 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/feature_importance.png')

---
## Step 9: Hyperparameter Tuning

Using GridSearchCV with 5-fold cross-validation, optimizing for **F1 Score** (best metric for imbalanced data).

In [ ]:
# 9a. Tune Logistic Regression
print('=== Tuning Logistic Regression ===')
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'class_weight': [None, 'balanced'],
    'solver': ['lbfgs'],
}
lr_tune = LogisticRegression(max_iter=2000, random_state=42)
lr_grid = GridSearchCV(lr_tune, lr_param_grid, scoring='f1', cv=5, n_jobs=-1, verbose=1)
lr_grid.fit(X_train_scaled, y_train)
print(f'Best params: {lr_grid.best_params_}')
print(f'Best CV F1: {lr_grid.best_score_:.4f}')
print(f'Test F1: {f1_score(y_test, lr_grid.predict(X_test_scaled)):.4f}')

In [ ]:
# 9b. Tune Random Forest
print('=== Tuning Random Forest ===')
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 10],
    'class_weight': [None, 'balanced'],
}
rf_tune = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid = GridSearchCV(rf_tune, rf_param_grid, scoring='f1', cv=5, n_jobs=-1, verbose=1)
rf_grid.fit(X_train_scaled, y_train)
print(f'Best params: {rf_grid.best_params_}')
print(f'Best CV F1: {rf_grid.best_score_:.4f}')
print(f'Test F1: {f1_score(y_test, rf_grid.predict(X_test_scaled)):.4f}')

In [ ]:
# 9c. Tune XGBoost
print('=== Tuning XGBoost ===')
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb_param_grid = {
    'learning_rate': [0.01, 0.1],
    'max_depth': [4, 6],
    'subsample': [0.8, 1.0],
    'scale_pos_weight': [1, scale_pos],
}
xgb_tune = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_grid = GridSearchCV(xgb_tune, xgb_param_grid, scoring='f1', cv=5, n_jobs=-1, verbose=1)
xgb_grid.fit(X_train_scaled, y_train)
print(f'Best params: {xgb_grid.best_params_}')
print(f'Best CV F1: {xgb_grid.best_score_:.4f}')
print(f'Test F1: {f1_score(y_test, xgb_grid.predict(X_test_scaled)):.4f}')

In [ ]:
# Save tuned models
joblib.dump(lr_grid.best_estimator_, MODELS_DIR + 'model_lr_tuned.pkl')
joblib.dump(rf_grid.best_estimator_, MODELS_DIR + 'model_rf_tuned.pkl')
joblib.dump(xgb_grid.best_estimator_, MODELS_DIR + 'model_xgb_tuned.pkl')
print('Tuned models saved to models/')

---
## Step 10: Final Model Comparison

In [ ]:
tuned_models = {
    'Logistic Regression (tuned)': lr_grid.best_estimator_,
    'Random Forest (tuned)': rf_grid.best_estimator_,
    'XGBoost (tuned)': xgb_grid.best_estimator_,
}

print('=' * 65)
print(f'{"Model":<32} {"Acc":>5} {"Prec":>5} {"Recall":>6} {"F1":>5} {"AUC":>5}')
print('=' * 65)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    print(f'{name:<32} {accuracy_score(y_test, y_pred):>5.3f} '
          f'{precision_score(y_test, y_pred):>5.3f} {recall_score(y_test, y_pred):>6.3f} '
          f'{f1_score(y_test, y_pred):>5.3f} {roc_auc_score(y_test, y_prob):>5.3f}')

---
## Step 11: SHAP Interpretation (XGBoost)

SHAP tells us **which features push a customer toward churn or retention** and by how much.

In [ ]:
xgb_best = tuned_models['XGBoost (tuned)']
explainer = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_test_scaled)

# SHAP Summary (direction + magnitude)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_scaled, show=False)
plt.tight_layout()
plt.savefig(ASSETS_DIR + 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/shap_summary.png')

In [ ]:
# SHAP Bar (global feature importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(ASSETS_DIR + 'shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: assets/shap_feature_importance.png')

In [ ]:
# Top 10 features by mean |SHAP|
mean_shap = np.abs(shap_values).mean(axis=0)
top_features = pd.DataFrame({
    'Feature': X_test_scaled.columns,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=False)

print('Top 10 features driving churn predictions:')
print(top_features.head(10).to_string(index=False))

---
## Step 12: Business Recommendations

### Key Drivers of Churn
1. **Contract_Two year** — strongly reduces churn (SHAP #1)
2. **tenure** — short tenure = high risk (SHAP #2)
3. **InternetService_Fiber optic** — Fiber users churn more than DSL
4. **Contract_One year** — better than month-to-month but not as protective as 2-year
5. **MonthlyCharges** — higher bills = higher churn risk

### Actionable Strategies

| Lever | Action |
|---|---|
| **Contract length** | Offer discounts for switching from month-to-month to annual |
| **Early tenure** | 'First 90 days' onboarding program with check-ins |
| **Fiber optic** | Investigate service quality, proactive outreach to Fiber users |
| **Electronic check** | Incentivize auto-pay setup with small monthly discount |
| **Risk scoring** | Score all customers monthly; top 20% get retention offers |

### Model Performance Summary

| Model | F1 | AUC | Best For |
|---|---|---|---|
| Logistic Regression | 0.613 | 0.841 | Interpretability (coefficients tell you why) |
| Random Forest | 0.627 | 0.839 | Robustness, handles outliers well |
| **XGBoost** | **0.627** | **0.842** | **Best overall** (tied F1, highest AUC) |

**Bottom line:** Deploy XGBoost to score all customers monthly. Focus retention offers on month-to-month contracts, short-tenure customers, Fiber users, and high-bill customers. This captures ~63% of potential churners (F1) and ranks churners vs non-churners with 84% accuracy (AUC).